# Main experiment on the electrocatalysis data

Same experiment as in `main_experiment_matrices.ipynb`, on a second dataset: 55 COF/CTF materials with a measured maximum diffusion current density (`JD_max`, negative values) from 27 papers. 46 COFs are made of two substrates; their SMILES are joined with '.', whole-substrate features cover both substrates and the main/side split is done for each substrate. 27 of the 55 materials have side chains.

| Experiment | Input to the featurizer |
|---|---|
| Substrate | whole substrate SMILES |
| Main + Side | main-chain and side-chain features concatenated (side all zeros without side chains) |
| Substrate + Main + Side | all three concatenated |

Featurizers: ECFP, RF best (EState + functional groups) and BRR best (custom descriptor + ECFP + EState + functional groups). Data from `scripts/prepare_electrocatalysis.py`, features from `configs/best_feature_generation.yaml` (`--target_col JD_max`), training with `scripts/models_training.py` and `configs/electrocatalysis.yaml` (10 folds). SMAPE of predicting the training median: 0.340.

In [1]:
import os

import joblib
import numpy as np
import pandas as pd

results_dir = "../results/electrocatalysis/"

FEATURE_SETS = {"ECFP": "ecfp", "RF best": "rf_best", "BRR best": "brr_best"}
EXPERIMENTS = {"Substrate": "substrate", "Main + Side": "main_side", "Substrate + Main + Side": "substrate_main_side"}
MODEL_DISPLAY_NAMES = {
    "bayes": "BRR",
    "knn": "KNN",
    "lasso": "LR",
    "mlp": "MLP",
    "random_forest": "RF",
    "xgboost": "XGB",
}

In [2]:
results = {"feature_set": [], "experiment": [], "model": [], "smape_mean": [], "smape_std": [], "pa_mean": [], "pa_std": []}

for feature_set, feature_set_name in FEATURE_SETS.items():
    for experiment, experiment_name in EXPERIMENTS.items():
        for model, model_name in MODEL_DISPLAY_NAMES.items():
            scores = joblib.load(os.path.join(results_dir, model, f"features_rth_0.99_{feature_set_name}_{experiment_name}", "scores.pickle"))
            results["feature_set"].append(feature_set)
            results["experiment"].append(experiment)
            results["model"].append(model_name)
            results["smape_mean"].append(np.mean(scores["smape"]))
            results["smape_std"].append(np.std(scores["smape"]))
            results["pa_mean"].append(np.mean(scores["pairwise_accuracy_score"]))
            results["pa_std"].append(np.std(scores["pairwise_accuracy_score"]))

df_results = pd.DataFrame(results)
df_results

,feature_set,experiment,model,smape_mean,smape_std,pa_mean,pa_std
0,ECFP,Substrate,BRR,0.315191,0.085603,0.740000,0.154056
1,ECFP,Substrate,KNN,0.329813,0.090628,0.640000,0.157621
2,ECFP,Substrate,LR,0.343634,0.089144,0.530000,0.045826
3,ECFP,Substrate,MLP,0.356852,0.118375,0.673333,0.205372
4,ECFP,Substrate,RF,0.316487,0.106022,0.676667,0.160589
5,ECFP,Substrate,XGB,0.314058,0.126931,0.690000,0.123873
6,ECFP,Main + Side,BRR,0.321772,0.086537,0.773333,0.145144
7,ECFP,Main + Side,KNN,0.331049,0.086808,0.620000,0.203961
8,ECFP,Main + Side,LR,0.359257,0.088917,0.500000,0.044721
9,ECFP,Main + Side,MLP,0.438038,0.178250,0.590000,0.194394


In [3]:
def to_matrix(values: str) -> pd.DataFrame:
    matrix = df_results.pivot(index=["feature_set", "experiment"], columns="model", values=values)
    rows = [(feature_set, experiment) for feature_set in FEATURE_SETS for experiment in EXPERIMENTS]
    return matrix.loc[rows, list(MODEL_DISPLAY_NAMES.values())]


def smape_pa_table(smape: pd.DataFrame, pa: pd.DataFrame) -> pd.DataFrame:
    return pd.concat({"SMAPE (lower is better)": smape, "PA (higher is better)": pa}, axis=1).round(3)

## SMAPE and PA

In [4]:
smape_pa_table(to_matrix("smape_mean"), to_matrix("pa_mean"))

SMAPE (lower is better)                \
model                                                   BRR    KNN     LR   
feature_set experiment                                                      
ECFP        Substrate                                 0.315  0.330  0.344   
            Main + Side                               0.322  0.331  0.359   
            Substrate + Main + Side                   0.318  0.324  0.352   
RF best     Substrate                                 0.331  0.331  0.352   
            Main + Side                               0.340  0.333  0.332   
            Substrate + Main + Side                   0.333  0.331  0.346   
BRR best    Substrate                                 0.319  0.303  0.349   
            Main + Side                               0.325  0.318  0.337   
            Substrate + Main + Side                   0.322  0.313  0.357   

                                                          \
model                                  MLP     RF    XGB   
feature_set experiment                                     
ECFP        Substrate                0.357  0.316  0.314   
            Main + Side              0.438  0.310  0.281   
            Substrate + Main + Side  0.433  0.315  0.293   
RF best     Substrate                0.487  0.268  0.256   
            Main + Side              0.334  0.303  0.276   
            Substrate + Main + Side  0.365  0.315  0.275   
BRR best    Substrate                0.327  0.327  0.279   
            Main + Side              0.395  0.326  0.291   
            Substrate + Main + Side  0.384  0.331  0.294   

                                    PA (higher is better)                \
model                                                 BRR    KNN     LR   
feature_set experiment                                                    
ECFP        Substrate                               0.740  0.640  0.530   
            Main + Side                             0.773  0.620  0.500   
            Substrate + Main + Side                 0.757  0.658  0.510   
RF best     Substrate                               0.650  0.590  0.550   
            Main + Side                             0.660  0.573  0.567   
            Substrate + Main + Side                 0.650  0.637  0.572   
BRR best    Substrate                               0.743  0.660  0.523   
            Main + Side                             0.747  0.633  0.567   
            Substrate + Main + Side                 0.717  0.633  0.520   

                                                          
model                                  MLP     RF    XGB  
feature_set experiment                                    
ECFP        Substrate                0.673  0.677  0.690  
            Main + Side              0.590  0.640  0.697  
            Substrate + Main + Side  0.640  0.630  0.727  
RF best     Substrate                0.627  0.690  0.737  
            Main + Side              0.643  0.682  0.730  
            Substrate + Main + Side  0.640  0.643  0.697  
BRR best    Substrate                0.743  0.633  0.707  
            Main + Side              0.627  0.660  0.707  
            Substrate + Main + Side  0.687  0.700  0.717